# 总体均值的估计完整教程：t 分布与置信区间

## 📚 教学目标
1. 理解 t 分布的特征和与正态分布的关系
2. 掌握均值的置信区间公式：$\bar{x} \pm t_{\alpha/2} \frac{s}{\sqrt{n}}$
3. 理解自由度的概念
4. 用 scipy.stats.t 计算临界值和置信区间
5. 理解样本量与置信区间宽度的关系

**参考书**：《基础统计学(第14版)》（Triola）第 7-2 节
**教学策略**：先用极小数据集让你「看见」t 分布和置信区间的每一步计算，再扩展到真实规模

---

## 1. 场景设定：为什么需要 t 分布？

### 🎯 一个直觉问题

我们知道总体均值 $\mu$ 的最佳点估计是样本均值 $\bar{x}$。

但问题是：**总体标准差 $\sigma$ 通常是未知的！**

- 如果 $\sigma$ 已知 → 用正态分布（z 分布）
- 如果 $\sigma$ 未知 → 用 **t 分布**（更常见的情况）

### 📖 书中的定义

> 样本均值 $\bar{x}$ 是总体均值 $\mu$ 的最佳点估计。即使是最好的点估计也不能告诉我们它有多精确，所以我们需要置信区间。

> 学生 t 分布通常被称为「t 分布」。威廉·戈塞（1876-1937）是 t 分布的提出者，当时他是吉尼斯啤酒厂的员工，他需要一个小样本的分布。虽然啤酒厂禁止发表研究结果，但他还是以「学生」的笔名发表了研究结果。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print('✅ 库导入完成')

---

## 2. t 分布 vs 正态分布

### 📐 t 分布的核心特征

t 分布与标准正态分布相似，但有以下区别：

| 特征 | 标准正态分布 | t 分布 |
|------|------------|--------|
| 均值 | 0 | 0 |
| 标准差 | 1 | > 1（取决于自由度） |
| 形状 | 钟形 | 钟形，但尾部更厚 |
| 参数 | 无 | 自由度 df |
| 小样本 | - | 更分散（更保守） |
| 大样本 | - | 趋近正态分布 |

### 💡 关键理解

t 分布的**自由度** $df = n - 1$，其中 $n$ 是样本量。

当 $n$ 很大时（如 $n > 30$），t 分布与正态分布几乎相同。

In [ ]:
# ========== 可视化：不同自由度的 t 分布 ==========
x = np.linspace(-4, 4, 1000)

fig, ax = plt.subplots(figsize=(12, 6))

# 标准正态分布
y_norm = stats.norm.pdf(x)
ax.plot(x, y_norm, 'k-', linewidth=2.5, label='Standard Normal (z)', alpha=0.8)

# 不同自由度的 t 分布
dfs = [1, 3, 5, 10, 30]
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

for df, color in zip(dfs, colors):
    y_t = stats.t.pdf(x, df)
    ax.plot(x, y_t, color=color, linewidth=1.5, linestyle='--', 
            label=f't dist (df={df})', alpha=0.7)

ax.set_xlabel('t / z Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('t Distribution vs Standard Normal Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.3)

# 添加注释
ax.annotate('t distribution has\nthicker tails\n(small df)', 
            xy=(-3, 0.02), xytext=(-3.8, 0.15),
            fontsize=10, ha='center',
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()

print(f'\n💡 图解说明：')
print(f'  t 分布（虚线）比正态分布（实线）有更厚的尾部')
print(f'  自由度越小，尾部越厚（更保守）')
print(f'  自由度增大时，t 分布趋近于标准正态分布')

---

## 3. 置信区间的核心公式

### 📐 总体均值的置信区间（$\sigma$ 未知）

$$\bar{x} - E < \mu < \bar{x} + E$$

其中误差范围 $E$ 为：

$$E = t_{\alpha/2} \frac{s}{\sqrt{n}}$$

变量说明：
- $\bar{x}$ = 样本均值（点估计）
- $s$ = 样本标准差
- $n$ = 样本量
- $t_{\alpha/2}$ = t 分布的临界值（自由度 $df = n-1$）
- $\alpha$ = 显著性水平 = 1 - 置信水平
- $E$ = 误差范围

### 📐 条件

1. 样本为简单随机样本
2. 总体服从正态分布，或 $n > 30$（中心极限定理）

### 📖 书中的要点

> 对于小样本的情况，即使偏离正态性，构建 $\mu$ 的置信区间的 t 方法也是具有稳健性的。此类分布不需要是完美的钟形曲线，但它应该满足：1. 样本数据的分布应该接近于轴对称；2. 样本数据的分布应该有一个众数；3. 样本数据不应包括任何异常值。

---

## 4. 微型数据手算：花生巧克力的重量

### 📖 书中例题

以下列出的是随机选取的瑞斯花生巧克力的重量（克）：

8.639, 8.689, 8.548, 8.980, 8.936, 9.042

每盒有 38 块巧克力，包装盒上写着总重量是 340.2 克，因此每块巧克力的平均重量应为 340.2/38 = 8.953 克。

**问题**：构建重量均值的 95% 置信区间。

In [ ]:
# ========== 微型数据集 ==========
weights = np.array([8.639, 8.689, 8.548, 8.980, 8.936, 9.042])
n_micro = len(weights)
confidence_level = 0.95

print('📋 微型数据集：瑞斯花生巧克力重量（克）')
print(f'  数据: {weights}')
print(f'  样本量 n = {n_micro}')
print(f'  置信水平 = {confidence_level*100:.0f}%')

In [ ]:
# ========== 步骤 1: 计算样本统计量 ==========
x_bar = np.mean(weights)
s_sample = np.std(weights, ddof=1)  # ddof=1 表示样本标准差

print(f'📊 步骤 1: 计算样本统计量')
print(f'  样本均值 x̄ = Σxᵢ / n = {np.sum(weights):.3f} / {n_micro} = {x_bar:.4f}')
print(f'  样本标准差 s = √[Σ(xᵢ - x̄)² / (n-1)] = {s_sample:.4f}')
print(f'  标准误 = s / √n = {s_sample:.4f} / √{n_micro} = {s_sample/np.sqrt(n_micro):.6f}')

In [ ]:
# ========== 步骤 2: 确定临界值 ==========
df_micro = n_micro - 1
alpha_micro = 1 - confidence_level
t_critical = stats.t.ppf(1 - alpha_micro / 2, df=df_micro)

print(f'📊 步骤 2: 确定 t 分布的临界值')
print(f'  自由度 df = n - 1 = {n_micro} - 1 = {df_micro}')
print(f'  α = {alpha_micro}, α/2 = {alpha_micro/2}')
print(f'  t_{{α/2, df={df_micro}}} = {t_critical:.4f}')
print(f'  💡 注意：比 z_{{0.025}} = 1.96 大，因为小样本需要更保守的估计')

In [ ]:
# ========== 步骤 3: 计算误差范围 ==========
se_micro = s_sample / np.sqrt(n_micro)
E_micro = t_critical * se_micro

print(f'📊 步骤 3: 计算误差范围 E')
print(f'  E = t_{{α/2}} × (s / √n)')
print(f'    = {t_critical:.4f} × ({s_sample:.4f} / √{n_micro})')
print(f'    = {t_critical:.4f} × {se_micro:.6f}')
print(f'    = {E_micro:.6f}')
print(f'  💡 误差范围 E = ±{E_micro:.4f} 克')

In [ ]:
# ========== 步骤 4: 构建置信区间 ==========
ci_lower = x_bar - E_micro
ci_upper = x_bar + E_micro

print(f'📊 步骤 4: 构建 {confidence_level*100:.0f}% 置信区间')
print(f'  x̂ - E < μ < x̂ + E')
print(f'  {x_bar:.4f} - {E_micro:.4f} < μ < {x_bar:.4f} + {E_micro:.4f}')
print(f'  {ci_lower:.4f} < μ < {ci_upper:.4f}')
print(f'  保留4位小数: {ci_lower:.4f} < μ < {ci_upper:.4f}')

print(f'\n🎯 解读:')
print(f'  我们有 95% 的把握认为，每块花生巧克力的平均重量在 {ci_lower:.4f} 克和 {ci_upper:.4f} 克之间')

# 检查是否包含标称值
nominal_weight = 340.2 / 38
print(f'\n📦 包装盒标称重量: {nominal_weight:.4f} 克/块')
if ci_lower <= nominal_weight <= ci_upper:
    print(f'  ✅ 置信区间包含标称值，每盒应该是按照标称重量填充的')
else:
    print(f'  ⚠️ 置信区间不包含标称值，实际重量可能与标称不符')

---

## 5. 用 scipy 验证手算结果

In [ ]:
# ========== 用 scipy 验证 ==========
# 方法1: 使用 scipy.stats.ttest_1samp（单样本 t 检验，但可以提取 CI）
# 方法2: 手动计算

# scipy 直接计算
t_stat, p_value = stats.ttest_1samp(weights, 0)  # 用 0 作为假设值，只为了得到统计量

# 计算置信区间
ci_scipy = stats.t.interval(confidence_level, df=df_micro, loc=x_bar, scale=se_micro)

print('🔬 scipy 验证:')
print(f'  手算 x̄ = {x_bar:.6f}')
print(f'  scipy x̄ = {np.mean(weights):.6f}')
print(f'  手算 s = {s_sample:.6f}')
print(f'  scipy s = {np.std(weights, ddof=1):.6f}')
print(f'  手算 t_{{critical}} = {t_critical:.6f}')
print(f'  scipy t_{{critical}} = {stats.t.ppf(1-alpha_micro/2, df_micro):.6f}')
print(f'  手算 CI = ({ci_lower:.6f}, {ci_upper:.6f})')
print(f'  scipy CI = ({ci_scipy[0]:.6f}, {ci_scipy[1]:.6f})')
print(f'  ✅ 验证通过！')

---

## 6. 扩展到大样本：500 个数据点

In [ ]:
# ========== 大样本模拟 ==========
# 模拟 500 块花生巧克力的重量
n_large = 500
mu_true = 8.953   # 真实均值（标称重量）
sigma_true = 0.2  # 假设的总体标准差

weights_large = np.random.normal(mu_true, sigma_true, n_large)

print('=' * 60)
print('📋 大样本数据集：500 块花生巧克力的重量')
print('=' * 60)
print(f'\n📊 数据生成过程 (DGP):')
print(f'  分布: Normal(μ={mu_true}, σ={sigma_true})')
print(f'  样本量: {n_large}')

# 计算统计量
x_bar_large = np.mean(weights_large)
s_large = np.std(weights_large, ddof=1)
df_large = n_large - 1
se_large = s_large / np.sqrt(n_large)
t_crit_large = stats.t.ppf(0.975, df_large)
E_large = t_crit_large * se_large

ci_lower_large = x_bar_large - E_large
ci_upper_large = x_bar_large + E_large

print(f'\n📊 样本统计量:')
print(f'  x̄ = {x_bar_large:.4f} (真实值: {mu_true})')
print(f'  s = {s_large:.4f} (真实值: {sigma_true})')
print(f'  df = {df_large}')
print(f'  标准误 = s/√n = {se_large:.6f}')

print(f'\n📊 95% 置信区间:')
print(f'  t_{{critical}} = {t_crit_large:.4f}')
print(f'  E = {E_large:.6f}')
print(f'  {ci_lower_large:.4f} < μ < {ci_upper_large:.4f}')

# 对比 z 分布
z_crit = stats.norm.ppf(0.975)
E_z = z_crit * se_large
print(f'\n🔬 对比 z 分布:')
print(f'  z_{{critical}} = {z_crit:.4f}')
print(f'  E (z) = {E_z:.6f}')
print(f'  差异 = {abs(E_large - E_z):.8f}')
print(f'  💡 大样本时，t 分布和 z 分布结果几乎相同')

In [ ]:
# ========== 可视化：大样本数据的分布 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：直方图 + 置信区间
ax1 = axes[0]
ax1.hist(weights_large, bins=30, density=True, alpha=0.6, color='steelblue', edgecolor='white')
x_plot = np.linspace(weights_large.min(), weights_large.max(), 100)
y_plot = stats.norm.pdf(x_plot, x_bar_large, s_large)
ax1.plot(x_plot, y_plot, 'r-', linewidth=2, label='Fitted Normal')
ax1.axvline(x=x_bar_large, color='green', linestyle='-', linewidth=2, label=f'x̄ = {x_bar_large:.4f}')
ax1.axvline(x=ci_lower_large, color='orange', linestyle='--', linewidth=1.5, label=f'95% CI')
ax1.axvline(x=ci_upper_large, color='orange', linestyle='--', linewidth=1.5)
ax1.axvline(x=mu_true, color='red', linestyle=':', linewidth=2, label=f'True μ = {mu_true}')
ax1.set_xlabel('Weight (g)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Distribution of Peanut Butter Cup Weights', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# 右图：QQ 图
ax2 = axes[1]
stats.probplot(weights_large, dist='norm', plot=ax2)
ax2.set_title('Normal Q-Q Plot', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n💡 图解说明：')
print(f'  左图：直方图显示数据近似正态分布，橙色虚线标示 95% 置信区间')
print(f'  右图：QQ 图的点接近直线，验证了正态性假设')

---

## 7. 选择正确的分布

### 📐 $\sigma$ 已知 vs $\sigma$ 未知

| 情况 | 条件 | 使用的分布 | 误差范围公式 |
|------|------|-----------|------------|
| $\sigma$ 已知 | 正态或 $n>30$ | 正态分布（z） | $E = z_{\alpha/2} \frac{\sigma}{\sqrt{n}}$ |
| $\sigma$ 未知 | 正态或 $n>30$ | t 分布 | $E = t_{\alpha/2} \frac{s}{\sqrt{n}}$ |
| $\sigma$ 未知 | 非正态且 $n<30$ | 自助法（7-4节） | 重抽样 |

### 💡 实际应用

在实际研究中，$\sigma$ 几乎总是未知的，所以**t 分布是最常用的方法**。

In [ ]:
# ========== 可视化：样本量对临界值的影响 ==========
sample_sizes = np.arange(2, 101)
t_crits = [stats.t.ppf(0.975, n-1) for n in sample_sizes]
z_crit = stats.norm.ppf(0.975)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sample_sizes, t_crits, 'b-', linewidth=2, label='t critical value')
ax.axhline(y=z_crit, color='red', linestyle='--', linewidth=1.5, alpha=0.7, 
           label=f'z critical = {z_crit:.4f}')

# 标记几个关键点
for n_mark in [5, 10, 20, 30, 50]:
    t_mark = stats.t.ppf(0.975, n_mark-1)
    ax.plot(n_mark, t_mark, 'ro', markersize=8, zorder=5)
    ax.annotate(f'n={n_mark}\nt={t_mark:.3f}', 
                xy=(n_mark, t_mark), xytext=(n_mark+3, t_mark+0.05),
                fontsize=9, arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Sample Size (n)', fontsize=12)
ax.set_ylabel('Critical Value', fontsize=12)
ax.set_title('t Critical Value vs Sample Size (95% Confidence)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(1.8, 13)

plt.tight_layout()
plt.show()

print(f'\n💡 图解说明：')
print(f'  小样本时，t 临界值远大于 z 临界值（更保守）')
print(f'  随着样本量增大，t 临界值趋近于 z 临界值 = 1.96')
print(f'  n > 30 后，差异已经很小')

---

## 8. 样本量的确定

### 📐 估计总体均值所需的样本量

$$n = \left(\frac{z_{\alpha/2} \cdot \sigma}{E}\right)^2$$

其中 $\sigma$ 未知时，可以用以下方法估计：

1. **范围经验法则**：$\sigma \approx \text{全距} / 4$
2. **使用之前的研究结果**
3. **保守估计**：使用一个较大的 $\sigma$ 值

In [ ]:
# ========== 样本量确定示例 ==========
# 估计统计学专业学生的平均智商分数
# 要求：95%置信水平，误差不超过3个IQ点
# 已知韦氏智商测试的标准差 σ = 15

sigma_iq = 15    # 韦氏智商测试的标准差
E_target = 3     # 目标误差范围（3个IQ点）
z_95 = stats.norm.ppf(0.975)  # 1.96

n_required = np.ceil((z_95 * sigma_iq / E_target) ** 2)

print('=' * 60)
print('📋 样本量确定：统计学专业学生的智商分数')
print('=' * 60)
print(f'\n  σ = {sigma_iq} (韦氏智商测试标准差)')
print(f'  E = {E_target} (目标误差范围)')
print(f'  z_{{α/2}} = {z_95:.4f} (95%置信水平)')
print(f'\n  n = (z_{{α/2}} × σ / E)²')
print(f'    = ({z_95:.4f} × {sigma_iq} / {E_target})²')
print(f'    = ({z_95 * sigma_iq / E_target:.4f})²')
print(f'    = {(z_95 * sigma_iq / E_target)**2:.2f}')
print(f'    = {int(n_required)} (向上取整)')
print(f'\n💡 需要至少 {int(n_required)} 名统计学专业学生的智商分数样本')

---

## 9. 综合案例：从原始数据到完整报告

In [ ]:
# ========== 完整报告示例 ==========

# 模拟：某工厂生产的螺丝钉长度（mm）
mu_screw = 25.0   # 真实均值
sigma_screw = 1.5  # 真实标准差
n_screw = 36       # 样本量

screws = np.random.normal(mu_screw, sigma_screw, n_screw)

# 计算统计量
x_bar_screw = np.mean(screws)
s_screw = np.std(screws, ddof=1)
df_screw = n_screw - 1
se_screw = s_screw / np.sqrt(n_screw)
t_crit_screw = stats.t.ppf(0.975, df_screw)
E_screw = t_crit_screw * se_screw

ci_low = x_bar_screw - E_screw
ci_up = x_bar_screw + E_screw

print('=' * 60)
print('📋 完整报告：螺丝钉长度的置信区间估计')
print('=' * 60)

print(f'\n🎯 研究问题:')
print(f'   估计某工厂生产的螺丝钉的平均长度')

print(f'\n📊 数据概况:')
print(f'   样本量: {n_screw}')
print(f'   样本均值: {x_bar_screw:.4f} mm')
print(f'   样本标准差: {s_screw:.4f} mm')
print(f'   标准误: {se_screw:.4f} mm')

print(f'\n🧮 统计推断:')
print(f'   置信水平: 95%')
print(f'   自由度: {df_screw}')
print(f'   临界值 t: {t_crit_screw:.4f}')
print(f'   误差范围: {E_screw:.4f} mm')
print(f'   95% 置信区间: ({ci_low:.4f}, {ci_up:.4f}) mm')

print(f'\n🎯 结论:')
if ci_low <= mu_screw <= ci_up:
    print(f'  ✓ 真实均值 μ = {mu_screw} 被包含在置信区间内')
else:
    print(f'  ✗ 真实均值 μ = {mu_screw} 未被包含在置信区间内')
print(f'  我们有 95% 的把握认为螺丝钉的平均长度在 {ci_low:.4f} mm 和 {ci_up:.4f} mm 之间')

print('\n' + '=' * 60)

---

## 📌 核心概念回顾

### 📌 t 分布 (Student's t Distribution)
- **定义**: 小样本下估计总体均值时使用的概率分布
- **参数**: 自由度 $df = n - 1$
- **含义**: 比正态分布有更厚的尾部，反映小样本的更大不确定性
- **判断标准**: $n > 30$ 时，t 分布与正态分布几乎相同

### 📌 自由度 (Degrees of Freedom)
- **定义**: 样本中可以自由变化的数据个数
- **公式**: $df = n - 1$
- **含义**: 对所有数据值做出均值的限制后，只有 $n-1$ 个值可以自由变化

### 📌 均值的置信区间
- **公式**: $\bar{x} \pm t_{\alpha/2} \frac{s}{\sqrt{n}}$
- **条件**: 简单随机样本 + (正态分布 或 $n>30$)
- **解读**: 「我们有 95% 的把握认为 $\mu$ 在此区间内」

### 📌 标准误 (Standard Error)
- **定义**: 样本均值的标准差
- **公式**: $SE = \frac{s}{\sqrt{n}}$
- **含义**: 衡量样本均值的精确度

### 🔗 完整流程
```
收集数据 → 计算统计量 → 确定自由度 → 查t临界值 → 计算E → 构建CI
    ↓           ↓            ↓           ↓          ↓        ↓
  n个数据    x̄, s          n-1        t_α/2,df    t×s/√n    x̄±E
```

### 📝 分布选择汇总

| $\sigma$ | 总体分布 | 样本量 | 使用分布 |
|---------|---------|--------|--------|
| 已知 | 正态 | 任意 | z 分布 |
| 已知 | 非正态 | $n>30$ | z 分布 |
| 未知 | 正态 | 任意 | t 分布 |
| 未知 | 非正态 | $n>30$ | t 分布 |
| 未知 | 非正态 | $n<30$ | 自助法 |

---

## ❌ 常见误区

### ❌ 误区 1: 「样本量小就不能用置信区间」
**✓ 正确理解**: 只要总体服从正态分布（或近似正态），即使样本量很小也可以用 t 分布构建置信区间。t 分布就是为小样本设计的。

### ❌ 误区 2: 「t 分布和 z 分布是一样的」
**✓ 正确理解**: t 分布有更厚的尾部，特别是小样本时差异显著。只有当 $n \to \infty$ 时，t 分布才趋近于标准正态分布。

### ❌ 误区 3: 「置信区间越窄越好，所以样本量越大越好」
**✓ 正确理解**: 置信区间确实越窄越好，但样本量增加的边际收益递减。要将误差减半，需要 4 倍的样本量。需要在精度和成本之间权衡。

### ❌ 误区 4: 「95% 置信区间意味着有 95% 的概率包含真实值」
**✓ 正确理解**: 真实值 $\mu$ 是固定参数，不是随机变量。95% 指的是如果重复抽样并构建区间，约 95% 的区间会包含真实值。

### ❌ 误区 5: 「用 s 代替 σ 不会影响结果」
**✓ 正确理解**: 用样本标准差 $s$ 代替总体标准差 $\sigma$ 会引入额外的不确定性，这正是需要使用 t 分布（而非 z 分布）的原因。t 分布的更厚尾部反映了这种额外不确定性。